<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="https://raw.githubusercontent.com/DataForScience/Networks/master/data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>Claude API for Python Developers</h1>
<h1>Code Generation and Explanation</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [1]:
import anthropic
from IPython.display import display, Markdown, Code

import matplotlib.pyplot as plt

import watermark

%load_ext watermark
%matplotlib inline

We start by print out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.13.3
IPython version      : 9.8.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 25.1.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: bb3251dc1b5c3edc5e3be2c5b1ada1ee0f046e1b

IPython   : 9.8.0
anthropic : 0.75.0
matplotlib: 3.10.7
watermark : 2.5.0



Load default figure style

In [3]:
plt.style.use('d4sci.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

Initialize the client and define utility functions

In [4]:
client = anthropic.Anthropic()

def show_response(response):
    """Display Claude's response with markdown formatting."""
    display(Markdown(response.content[0].text))

def show_json(content):
    print(json.dumps(content, indent=2))

def show_code(response, language="python"):
    """Display code with syntax highlighting."""
    display(Markdown(f"```{language}\n{response.content[0].text}\n```"))

## Generating Code from Prompts

Claude excels at generating code from natural language descriptions. The key is providing clear, specific requirements.

In [5]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    system="You are an expert Python developer. Write clean, well-documented pythonic code.",
    messages=[
        {
            "role": "user",
            "content": "Write a Python function that checks if a string is a valid email address using regex."
        }
    ]
)

In [6]:
show_response(response)

Here's a Python function that validates email addresses using regex:

```python
import re
from typing import Union


def is_valid_email(email: str) -> bool:
    """
    Check if a string is a valid email address using regex.
    
    This function implements a comprehensive email validation that covers
    most common email formats while being practical for real-world use.
    
    Args:
        email (str): The email address string to validate
        
    Returns:
        bool: True if the email is valid, False otherwise
        
    Examples:
        >>> is_valid_email("user@example.com")
        True
        >>> is_valid_email("test.email+tag@domain.co.uk")
        True
        >>> is_valid_email("invalid.email")
        False
        >>> is_valid_email("@domain.com")
        False
    """
    if not isinstance(email, str):
        return False
    
    # Comprehensive email regex pattern
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    
    # Additional length checks (RFC 5321 limits)
    if len(email) > 254:  # Maximum email length
        return False
    
    local_part, _, domain_part = email.rpartition('@')
    if len(local_part) > 64:  # Maximum local part length
        return False
    
    return bool(re.match(pattern, email))


def is_valid_email_strict(email: str) -> bool:
    """
    A more strict email validation with additional checks.
    
    This version includes more rigorous validation rules:
    - No consecutive dots
    - No leading/trailing dots in local part
    - Domain must have at least one dot
    - More restrictive character set
    
    Args:
        email (str): The email address string to validate
        
    Returns:
        bool: True if the email is valid under strict rules, False otherwise
        
    Examples:
        >>> is_valid_email_strict("user@example.com")
        True
        >>> is_valid_email_strict("user..name@example.com")
        False
        >>> is_valid_email_strict(".user@example.com")
        False
    """
    if not isinstance(email, str):
        return False
    
    # More strict pattern
    pattern = r'^[a-zA-Z0-9]([a-zA-Z0-9._-]*[a-zA-Z0-9])?@[a-zA-Z0-9]([a-zA-Z0-9.-]*[a-zA-Z0-9])?\.[a-zA-Z]{2,}$'
    
    # Length checks
    if len(email) > 254:
        return False
    
    if '@' not in email:
        return False
    
    local_part, domain_part = email.rsplit('@', 1)
    
    if len(local_part) > 64:
        return False
    
    # Check for consecutive dots
    if '..' in email:
        return False
    
    # Check if local part starts or ends with dot
    if local_part.startswith('.') or local_part.endswith('.'):
        return False
    
    # Check if domain starts or ends with dot or hyphen
    if (domain_part.startswith('.') or domain_part.endswith('.') or 
        domain_part.startswith('-') or domain_part.endswith('-')):
        return False
    
    return bool(re.match(pattern, email))


# Example usage and test cases
if __name__ == "__main__":
    # Test cases
    test_emails = [
        ("user@example.com", True),
        ("test.email+tag@domain.co.uk", True),
        ("user123@test-domain.org", True),
        ("name.lastname@company.travel", True),
        ("invalid.email", False),
        ("@domain.com", False),
        ("user@", False),
        ("user@domain", False),
        ("user..name@example.com", True),  # Valid in basic, invalid in strict
        (".user@

The more specific the requirements/prompt the better the outcome

In [7]:
detailed_prompt = """
Write a Python class called `RateLimiter` with the following specifications:

1. Constructor takes `max_requests` (int) and `time_window` (int, seconds)
2. Method `is_allowed()` returns True if request is within rate limit
3. Method `reset()` clears the request history
4. Use a sliding window algorithm
5. Include type hints
6. Add docstrings

Example usage:
```python
limiter = RateLimiter(max_requests=5, time_window=60)
if limiter.is_allowed():
    # Process request
```
"""

In [8]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    system="You are an expert Python developer. Write production-quality code.",
    messages=[{"role": "user", "content": detailed_prompt}]
)

show_response(response)

```python
import time
from collections import deque
from typing import Deque


class RateLimiter:
    """
    A rate limiter implementation using a sliding window algorithm.
    
    This class tracks requests within a specified time window and enforces
    a maximum number of requests per time period.
    
    Attributes:
        max_requests (int): Maximum number of requests allowed within the time window
        time_window (int): Time window in seconds
    """
    
    def __init__(self, max_requests: int, time_window: int) -> None:
        """
        Initialize the RateLimiter.
        
        Args:
            max_requests (int): Maximum number of requests allowed within the time window.
                               Must be a positive integer.
            time_window (int): Time window in seconds. Must be a positive integer.
            
        Raises:
            ValueError: If max_requests or time_window is not positive.
        """
        if max_requests <= 0:
            raise ValueError("max_requests must be a positive integer")
        if time_window <= 0:
            raise ValueError("time_window must be a positive integer")
            
        self.max_requests = max_requests
        self.time_window = time_window
        self._request_timestamps: Deque[float] = deque()
    
    def is_allowed(self) -> bool:
        """
        Check if a request is allowed based on the rate limit.
        
        This method implements a sliding window algorithm by:
        1. Getting the current timestamp
        2. Removing expired timestamps outside the time window
        3. Checking if we can accept a new request
        4. If allowed, recording the current timestamp
        
        Returns:
            bool: True if the request is allowed, False otherwise.
        """
        current_time = time.time()
        
        # Remove timestamps outside the current time window
        self._remove_expired_requests(current_time)
        
        # Check if we can accept a new request
        if len(self._request_timestamps) < self.max_requests:
            # Record this request
            self._request_timestamps.append(current_time)
            return True
        
        return False
    
    def reset(self) -> None:
        """
        Clear the request history.
        
        This method removes all recorded request timestamps, effectively
        resetting the rate limiter state.
        """
        self._request_timestamps.clear()
    
    def _remove_expired_requests(self, current_time: float) -> None:
        """
        Remove request timestamps that are outside the current time window.
        
        Args:
            current_time (float): The current timestamp to compare against.
        """
        cutoff_time = current_time - self.time_window
        
        # Remove timestamps older than the time window
        while self._request_timestamps and self._request_timestamps[0] <= cutoff_time:
            self._request_timestamps.popleft()
    
    def get_remaining_requests(self) -> int:
        """
        Get the number of requests remaining in the current time window.
        
        Returns:
            int: Number of requests that can still be made without being rate limited.
        """
        current_time = time.time()
        self._remove_expired_requests(current_time)
        return max(0, self.max_requests - len(self._request_timestamps))
    
    def get_reset_time(self) -> float:
        """
        Get the timestamp when the rate limit will reset (oldest request expires).
        
        Returns:
            float: Timestamp when the next request slot becomes available,
                   or current time if there are available slots.
        """
        if not self._request_timestamps:
            return time.time()
        
        # The rate limit resets when the oldest request expires
        return self._request_timestamps[0] + self.time_window
    
    def __repr__(self) -> str:
        """
        Return a string representation of the RateLimiter.
        
        Returns:
            str: String representation including configuration and current state.
        """
        return (f"RateLimiter(max_requests={self.max_requests}, "
                f"time_window={self.time_window}, "
                f"current_requests={len(self._request_timestamps)})")


# Example usage and testing
if __name__ == "__main__":
    # Basic usage example
    limiter = RateLimiter(max_requests=5, time_window=60)
    
    print(f"Rate limiter created: {limiter}")
    print(f"Remaining requests: {limiter.get_remaining_requests()}")
    
    # Test multiple requests
    for i in range(7):
        if limiter.is_allowed():
            print(f"Request {i + 1}: Allowed")
        else:
            print(f"Request {i + 1}: Rate limited")
    
    print(f"Remaining requests: {limiter.get_remaining_requests()}")
    print(f"Reset time: {limiter.get_reset_time()}")
    
    # Reset and try again
    limiter.reset()
    print(f"\nAfter reset - Remaining requests: {limiter.get_remaining_requests()}")
```

This implementation provides:

1. **Sliding Window Algorithm**: Uses a `deque` to efficiently track request timestamps and removes expired entries
2. **Type Hints**: All methods and attributes are properly type-hinted
3. **Comprehensive Docstrings**: Each method includes detailed documentation
4. **Input Validation**: Constructor validates that parameters are positive
5. **Additional Utility Methods**:
   - `get_remaining_requests()`: Shows how many requests are left
   - `get_reset_time()`: Shows when the rate limit will reset
   - `__repr__()`: Provides a useful string representation

6. **Thread Safety Consideration**: While this basic implementation isn't thread-safe, it can be made thread-safe by adding locks around the critical sections if needed for concurrent usage.

7. **Efficient Implementation**: Uses `deque.popleft()` for O(1) removal of expired timestamps

The sliding window approach ensures accurate rate limiting by only considering requests within the exact time window, providing smooth and predictable rate limiting behavior.

### Generating Code in Multiple Languages


Claude can generate code in many languages

In [9]:
languages = ["Python", "JavaScript", "TypeScript", "Rust", "Go"]

task = "Write a function to reverse a string"

print("Same task in multiple languages:\n")

for lang in languages:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=300,
        messages=[{
            "role": "user",
            "content": f"{task}. Use {lang}. Just the code, no explanation."
        }]
    )
    
    print(f"{lang}:")
    show_response(response)
    print("\n" + "-"*50 + "\n")

Same task in multiple languages:

Python:


```python
def reverse_string(s):
    return s[::-1]
```


--------------------------------------------------

JavaScript:


```javascript
function reverseString(str) {
    return str.split('').reverse().join('');
}
```


--------------------------------------------------

TypeScript:


...


--------------------------------------------------

Rust:


```rust
fn reverse_string(s: &str) -> String {
    s.chars().rev().collect()
}
```


--------------------------------------------------

Go:


```go
func reverseString(s string) string {
    runes := []rune(s)
    for i, j := 0, len(runes)-1; i < j; i, j = i+1, j-1 {
        runes[i], runes[j] = runes[j], runes[i]
    }
    return string(runes)
}
```


--------------------------------------------------



### SQL queries from natural language

In [11]:
sql_prompt = """
I have a database with the following tables:
    - users (id, name, email, created_at, subscription_tier)
    - orders (id, user_id, total_amount, status, created_at)
    - products (id, name, price, category)
    - order_items (order_id, product_id, quantity)
    
Write SQL queries for:
    1. Get top 10 customers by total spending
    2. Find products that have never been ordered
    3. Calculate monthly revenue for the past year
    4. Find users who haven't ordered in the last 30 days
"""

In [12]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    system="You are a database expert. Write efficient, readable SQL queries.",
    messages=[{"role": "user", "content": sql_prompt}]
)

show_response(response)

I'll provide efficient SQL queries for each of your requirements:

## 1. Top 10 customers by total spending

```sql
SELECT 
    u.id,
    u.name,
    u.email,
    SUM(o.total_amount) as total_spent
FROM users u
INNER JOIN orders o ON u.id = o.user_id
WHERE o.status != 'cancelled'  -- Assuming cancelled orders shouldn't count
GROUP BY u.id, u.name, u.email
ORDER BY total_spent DESC
LIMIT 10;
```

## 2. Products that have never been ordered

```sql
SELECT 
    p.id,
    p.name,
    p.price,
    p.category
FROM products p
LEFT JOIN order_items oi ON p.id = oi.product_id
WHERE oi.product_id IS NULL
ORDER BY p.name;
```

## 3. Monthly revenue for the past year

```sql
SELECT 
    DATE_TRUNC('month', o.created_at) as month,
    SUM(o.total_amount) as monthly_revenue,
    COUNT(o.id) as order_count
FROM orders o
WHERE o.created_at >= CURRENT_DATE - INTERVAL '1 year'
    AND o.status != 'cancelled'  -- Exclude cancelled orders
GROUP BY DATE_TRUNC('month', o.created_at)
ORDER BY month DESC;
```

*Alternative for MySQL (if DATE_TRUNC is not available):*
```sql
SELECT 
    DATE_FORMAT(o.created_at, '%Y-%m') as month,
    SUM(o.total_amount) as monthly_revenue,
    COUNT(o.id) as order_count
FROM orders o
WHERE o.created_at >= DATE_SUB(CURRENT_DATE, INTERVAL 1 YEAR)
    AND o.status != 'cancelled'
GROUP BY DATE_FORMAT(o.created_at, '%Y-%m')
ORDER BY month DESC;
```

## 4. Users who haven't ordered in the last 30 days

```sql
SELECT 
    u.id,
    u.name,
    u.email,
    u.subscription_tier,
    MAX(o.created_at) as last_order_date
FROM users u
LEFT JOIN orders o ON u.id = o.user_id
GROUP BY u.id, u.name, u.email, u.subscription_tier
HAVING MAX(o.created_at) < CURRENT_DATE - INTERVAL '30 days'
    OR MAX(o.created_at) IS NULL  -- Users who never ordered
ORDER BY last_order_date DESC NULLS LAST;
```

## Additional Performance Notes:

1. **Indexes recommended** for optimal performance:
   - `orders(user_id, created_at, status)`
   - `order_items(product_id)`
   - `orders(created_at, status)`

2. **Alternative approach for query #4** if you only want users with previous orders:
```sql
SELECT 
    u.id,
    u.name,
    u.email,
    u.subscription_tier,
    MAX(o.created_at) as last_order_date
FROM users u
INNER JOIN orders o ON u.id = o.user_id
GROUP BY u.id, u.name, u.email, u.subscription_tier
HAVING MAX(o.created_at) < CURRENT_DATE - INTERVAL '30 days'
ORDER BY last_order_date DESC;
```

These queries are optimized for readability and performance, with appropriate filtering for business logic (like excluding cancelled orders where relevant).

## Explaining Existing Code

Some code to explain

In [13]:
complex_code = '''
def memoize(func):
    cache = {}
    def wrapper(*args, **kwargs):
        key = str(args) + str(sorted(kwargs.items()))
        if key not in cache:
            cache[key] = func(*args, **kwargs)
        return cache[key]
    return wrapper

@memoize
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)
'''

In [14]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{
        "role": "user",
        "content": f"""Explain this Python code in detail. Cover:
1. What each part does
2. How the decorator pattern works
3. Why memoization helps here
4. The time complexity improvement

```python
{complex_code}
```"""
    }]
)

In [15]:
show_response(response)

I'll break down this memoization implementation and explain how it dramatically improves the Fibonacci calculation.

## 1. What Each Part Does

### The Memoize Decorator Function
```python
def memoize(func):
    cache = {}  # Dictionary to store previously computed results
    def wrapper(*args, **kwargs):
        # Create a unique key from function arguments
        key = str(args) + str(sorted(kwargs.items()))
        
        if key not in cache:
            # If result not cached, compute and store it
            cache[key] = func(*args, **kwargs)
        
        # Return cached result
        return cache[key]
    return wrapper
```

**Key components:**
- `cache = {}`: A dictionary that persists between function calls
- `key = str(args) + str(sorted(kwargs.items()))`: Creates a string representation of all arguments to use as a cache key
- The cache check: Only computes the function if the result isn't already stored

### The Fibonacci Function
```python
@memoize
def fibonacci(n):
    if n < 2:
        return n  # Base cases: fib(0)=0, fib(1)=1
    return fibonacci(n - 1) + fibonacci(n - 2)
```

This is the classic recursive Fibonacci implementation, but now enhanced with memoization.

## 2. How the Decorator Pattern Works

```python
# When you write:
@memoize
def fibonacci(n):
    # function body

# Python effectively does this:
def fibonacci(n):
    # function body
fibonacci = memoize(fibonacci)
```

**The process:**
1. `memoize(fibonacci)` is called, receiving the original function
2. Returns the `wrapper` function with access to `cache` via closure
3. `fibonacci` now refers to the `wrapper` function
4. Every call to `fibonacci()` actually calls `wrapper()`

## 3. Why Memoization Helps Here

### The Problem with Naive Fibonacci
Without memoization, calculating `fibonacci(5)` looks like this:

```
fibonacci(5)
├── fibonacci(4)
│   ├── fibonacci(3)
│   │   ├── fibonacci(2)
│   │   │   ├── fibonacci(1) → 1
│   │   │   └── fibonacci(0) → 0
│   │   └── fibonacci(1) → 1
│   └── fibonacci(2)
│       ├── fibonacci(1) → 1  # Redundant calculation!
│       └── fibonacci(0) → 0  # Redundant calculation!
└── fibonacci(3)  # Entire subtree calculated again!
    ├── fibonacci(2)
    │   ├── fibonacci(1) → 1
    │   └── fibonacci(0) → 0
    └── fibonacci(1) → 1
```

Notice how `fibonacci(2)`, `fibonacci(1)`, etc. are calculated multiple times.

### With Memoization
```python
# First call: fibonacci(3)
print(fibonacci(3))  # Computes and caches fib(0), fib(1), fib(2), fib(3)

# Second call: fibonacci(4) 
print(fibonacci(4))  # Only computes fib(4), reuses cached values

# The cache now contains:
# {'(0,)': 0, '(1,)': 1, '(2,)': 1, '(3,)': 2, '(4,)': 3}
```

## 4. Time Complexity Improvement

### Without Memoization: O(2^n)
- Each call spawns two recursive calls
- Results in exponential growth
- `fibonacci(40)` requires ~2.7 billion function calls

### With Memoization: O(n)
- Each value from 0 to n is computed exactly once
- Subsequent lookups are O(1)
- `fibonacci(40)` requires only 40 function calls

### Practical Demonstration
```python
import time

# Without memoization - this would be very slow for n=35
def fib_slow(n):
    if

Explaining code for different audiences

In [16]:
async_code = '''
    async def fetch_all(urls):
        async with aiohttp.ClientSession() as session:
            tasks = [fetch_one(session, url) for url in urls]
            return await asyncio.gather(*tasks)
    
    async def fetch_one(session, url):
        async with session.get(url) as response:
            return await response.json()
'''

In [18]:
audiences = [
    ("beginner",     "Explain to someone new to programming. Use simple analogies."),
    ("intermediate", "Explain to someone familiar with Python but new to async."),
    ("expert",       "Focus on performance implications and best practices.")
]

print("Same code, different audiences:\n")

for level, instruction in audiences:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": f"""{instruction}

```python
{async_code}
```"""
        }]
    )
    
    print(f"  {level.upper()} Explanation:")
    print(response.content[0].text[:500] + "..." if len(response.content[0].text) > 500 else response.content[0].text)
    print("\n" + "="*60 + "\n")

Same code, different audiences:

  BEGINNER Explanation:
Great question! Let me break this down using simple analogies that make async programming easy to understand.

## The Restaurant Analogy 🍽️

Imagine you're a **waiter at a busy restaurant** who needs to get food from the kitchen for multiple tables:

### **Old Way (Synchronous):**
- Go to kitchen for Table 1's order
- **Wait** until chef finishes cooking
- Carry food back to Table 1
- **Then** go to kitchen for Table 2's order
- **Wait** again until chef finishes
- And so on...

This is slow bec...


  INTERMEDIATE Explanation:
Let me break down this async code for you! This is a common pattern for making multiple HTTP requests concurrently.

## The Big Picture
Instead of fetching URLs one by one (which would be slow), this code fetches them all at the same time. Think of it like opening multiple browser tabs simultaneously rather than waiting for each page to load before opening the next.

## Key Concepts

**`async def`** - Mark

Review code for vulnerabilities and other issues

In [19]:
code_to_review = '''
def get_user_data(user_id):
    connection = mysql.connector.connect(host="localhost", user="root", password="password123", database="mydb")
    cursor = connection.cursor()
    cursor.execute("SELECT * FROM users WHERE id = " + str(user_id))
    result = cursor.fetchone()
    return result
'''

In [20]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1200,
    messages=[{
        "role": "user",
        "content": f"""Review this code for:
1. Security vulnerabilities
2. Best practices violations
3. Error handling issues
4. Provide an improved version

```python
{code_to_review}
```"""
    }]
)

show_response(response)

I've identified several critical security vulnerabilities and best practices violations in this code. Let me break them down:

## 🚨 Security Vulnerabilities

1. **SQL Injection (Critical)** - String concatenation creates vulnerability to SQL injection attacks
2. **Hardcoded Credentials (Critical)** - Database password exposed in source code
3. **Information Disclosure** - Returns all user columns, potentially exposing sensitive data

## ❌ Best Practices Violations

1. **No resource management** - Database connections not properly closed
2. **No error handling** - Silent failures on connection/query errors
3. **No input validation** - Accepts any user_id without verification
4. **Overprivileged access** - Using root database user

## 🔧 Improved Version

Here's a secure, production-ready implementation:

```python
import mysql.connector
import os
import logging
from typing import Optional, Dict, Any
from contextlib import contextmanager

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class DatabaseError(Exception):
    """Custom exception for database operations"""
    pass

class UserNotFoundError(Exception):
    """Raised when user is not found"""
    pass

@contextmanager
def get_db_connection():
    """Context manager for database connections with proper cleanup"""
    connection = None
    try:
        connection = mysql.connector.connect(
            host=os.getenv('DB_HOST', 'localhost'),
            user=os.getenv('DB_USER', 'app_user'),  # Use dedicated app user
            password=os.getenv('DB_PASSWORD'),       # From environment
            database=os.getenv('DB_NAME', 'mydb'),
            port=int(os.getenv('DB_PORT', 3306)),
            autocommit=False,
            charset='utf8mb4',
            collation='utf8mb4_unicode_ci'
        )
        yield connection
    except mysql.connector.Error as e:
        logger.error(f"Database connection failed: {e}")
        raise DatabaseError(f"Failed to connect to database: {e}")
    finally:
        if connection and connection.is_connected():
            connection.close()

def validate_user_id(user_id: Any) -> int:
    """Validate and convert user_id to integer"""
    try:
        user_id_int = int(user_id)
        if user_id_int <= 0:
            raise ValueError("User ID must be positive")
        return user_id_int
    except (ValueError, TypeError) as e:
        raise ValueError(f"Invalid user ID: {user_id}") from e

def get_user_data(user_id: Any) -> Optional[Dict[str, Any]]:
    """
    Retrieve user data by ID with proper security and error handling.
    
    Args:
        user_id: The user ID to lookup
        
    Returns:
        Dictionary containing user data or None if not found
        
    Raises:
        ValueError: If user_id is invalid
        DatabaseError: If database operation fails
    """
    # Input validation
    try:
        validated_user_id = validate_user_id(user_id)
    except ValueError as e:
        logger.warning(f"Invalid user ID provided: {user_id}")
        raise
    
    # Define specific columns to return (principle of least privilege)
    query = """
        SELECT id, username, email, first_name, last_name, created_at, last_login
        FROM users 
        WHERE id = %s AND is_active = 1
    """
    
    try:
        with get_db_connection() as connection:
            cursor = connection.cursor(dictionary=True)  # Return dict instead of tuple
            
            # Use parameterized query to prevent SQL injection
            cursor.execute(query, (validated_user_id,))
            result = cursor.fetchone()
            
            if result:
                logger.info(f"User data retrieved for ID: {validated_user_id}")
                return result
            else:
                logger.info(f"User not found for ID: {validated_user_id}")
                return None
                
    except mysql.connector.Error as e:
        logger.error(f"Database query failed for user ID {validated_user_id}: {e}")
        raise DatabaseError(f"Failed to retrieve user data: {e}")
    except Exception as e:
        logger.error(f"Unexpected error retrieving user {validated_user_id}: {e}")
        raise DatabaseError(f"Unexpected error: {e}")

# Alternative version with connection pooling for high-traffic applications
class UserRepository:
    """Repository class with connection pooling for better performance"""
    
    def __init__(self):
        self.pool = mysql.connector.pooling.MySQLConnectionPool(
            pool_name="user_pool",
            pool_size=10,
            pool_reset_session=True,
            host=os.getenv('

## Generating Comments and Documentation

Claude can add documentation at various levels: inline comments, docstrings, and full documentation.

In [21]:
undocumented_code = '''
class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.data = None
        self._cache = {}
    
    def load(self, filepath):
        with open(filepath, 'r') as f:
            self.data = json.load(f)
        return self
    
    def transform(self, operations):
        if self.data is None:
            raise ValueError("No data loaded")
        
        result = self.data.copy()
        for op in operations:
            op_name = op['type']
            if op_name in self._cache:
                result = self._cache[op_name](result, op.get('params', {}))
            else:
                handler = getattr(self, f'_op_{op_name}', None)
                if handler:
                    self._cache[op_name] = handler
                    result = handler(result, op.get('params', {}))
        return result
    
    def _op_filter(self, data, params):
        field = params['field']
        value = params['value']
        return [d for d in data if d.get(field) == value]
    
    def _op_sort(self, data, params):
        return sorted(data, key=lambda x: x.get(params['field'], ''))
'''

In [22]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=2000,
    messages=[{
        "role": "user",
        "content": f"""Add comprehensive documentation to this code:
1. Class-level docstring (purpose, usage example)
2. Method docstrings (Google style with Args, Returns, Raises)
3. Inline comments for complex logic
4. Type hints

```python
{undocumented_code}
```

Return the fully documented code."""
    }]
)

show_response(response)

```python
import json
from typing import Dict, List, Any, Union, Callable, Optional


class DataProcessor:
    """A flexible data processing utility for loading, transforming, and manipulating JSON data.
    
    This class provides a fluent interface for loading JSON data from files and applying
    a series of transformations using configurable operations. It supports caching of
    operation handlers for improved performance on repeated operations.
    
    The processor supports various operations like filtering and sorting, and can be
    extended with custom operations by adding methods with the '_op_' prefix.
    
    Attributes:
        config (Dict[str, Any]): Configuration dictionary for the processor.
        data (Optional[List[Dict[str, Any]]]): The loaded data, None if no data is loaded.
    
    Example:
        >>> config = {'encoding': 'utf-8', 'strict_mode': True}
        >>> processor = DataProcessor(config)
        >>> 
        >>> # Load and transform data in a fluent interface
        >>> operations = [
        ...     {'type': 'filter', 'params': {'field': 'status', 'value': 'active'}},
        ...     {'type': 'sort', 'params': {'field': 'name'}}
        ... ]
        >>> result = processor.load('data.json').transform(operations)
        >>> print(result)
    """
    
    def __init__(self, config: Dict[str, Any]) -> None:
        """Initialize the DataProcessor with configuration settings.
        
        Args:
            config: A dictionary containing configuration options for the processor.
                   Can include settings like encoding, strict_mode, etc.
        """
        self.config: Dict[str, Any] = config
        self.data: Optional[List[Dict[str, Any]]] = None
        # Cache for operation handlers to avoid repeated getattr lookups
        self._cache: Dict[str, Callable] = {}

    def load(self, filepath: str) -> 'DataProcessor':
        """Load JSON data from a file.
        
        Loads JSON data from the specified file path and stores it in the instance.
        This method supports method chaining by returning self.
        
        Args:
            filepath: Path to the JSON file to load. Must be a valid file path
                     containing properly formatted JSON data.
        
        Returns:
            DataProcessor: Returns self to enable method chaining.
            
        Raises:
            FileNotFoundError: If the specified file doesn't exist.
            json.JSONDecodeError: If the file contains invalid JSON.
            IOError: If there are permission issues or other I/O errors.
            
        Example:
            >>> processor = DataProcessor({})
            >>> processor.load('users.json')
            >>> print(len(processor.data))  # Number of loaded records
        """
        with open(filepath, 'r') as f:
            self.data = json.load(f)
        return self

    def transform(self, operations: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Apply a series of transformations to the loaded data.
        
        Executes a sequence of operations on the data. Each operation is defined
        by a dictionary with 'type' and optional 'params' keys. Operations are
        cached after first use for improved performance.
        
        Args:
            operations: A list of operation dictionaries. Each dictionary must contain:
                       - 'type' (str): The operation name (e.g., 'filter', 'sort')
                       - 'params' (Dict, optional): Parameters for the operation
                       
        Returns:
            List[Dict[str, Any]]: The transformed data as a list of dictionaries.
            
        Raises:
            ValueError: If no data has been loaded (data is None).
            AttributeError: If an operation type doesn't have a corresponding handler method.
            
        Example:
            >>> operations = [
            ...     {'type': 'filter', 'params': {'field': 'age', 'value': 25}},
            ...     {'type': 'sort', 'params': {'field': 'name'}}
            ... ]
            >>> result = processor.transform(operations)
        """
        if self.data is None:
            raise ValueError("No data loaded")

        # Work with a copy to avoid modifying the original data
        result = self.data.copy()
        
        for op in operations:
            op_name = op['type']
            
            # Check cache first for performance optimization
            if op_name in self._cache:
                result = self._cache[op_name](result, op.get('params', {}))
            else:
                # Dynamically find the operation handler method
                # Handler methods follow the pattern '_op_{operation_name}'
                handler = getattr(self, f'_op_{op_name}', None)
                if handler:
                    # Cache the handler for future use
                    self._cache[op_name] = handler
                    result = handler(result, op.get('params', {}))
                else:
                    raise AttributeError(f"No handler found for operation: {op_name}")
                    
        return result

    def _op_filter(self, data: List[Dict[str, Any]], params: Dict[str, Any]) -> List[Dict[str, Any]]:
        """Filter operation handler that filters data based on field value equality.
        
        Filters the input data to include only records where the specified field
        matches the specified value exactly.
        
        Args:
            data: List of dictionaries to filter.
            params: Dictionary containing:
                   - 'field' (str): The field name to filter on
                   - 'value' (Any): The value to match against
                   
        Returns:
            List[Dict[str, Any]]: Filtered list containing only matching records.
            
        Example:
            >>> data = [{'name': 'Alice', 'status': 'active'}, {'name': 'Bob', 'status': 'inactive'}]
            >>> params = {'field': 'status', 'value': 'active'}
            >>> result = processor._op_filter(data, params)
            >>> # Returns: [{'name': 'Alice', 'status': 'active'}]
        """
        field = params['field']
        value = params['value']
        # Use .get() with field name to safely handle missing fields
        return [d for d in data if d.get(field) == value]

    def _op_sort(self, data: List[Dict[str, Any]], params: Dict[str, Any]) -> List[Dict[str, Any]]:
        """Sort operation handler that sorts data by a specified field.
        
        Sorts the input data in ascending order based on the specified field.
        Missing fields are treated as empty strings for sorting purposes.
        
        Args:
            data: List of dictionaries to sort.
            params: Dictionary containing:
                   - 'field' (str): The field name to sort by
                   
        Returns:
            List[Dict[str, Any]]: New sorted list (original data unchanged).
            
        Example:
            >>> data = [{'name': 'Charlie'}, {'name': 'Alice'}, {'name': 'Bob'}]
            >>> params = {'field': 'name'}
            >>> result = processor._op_sort(data, params)
            >>> # Returns data sorted by name: Alice, Bob, Charlie
        """
        # Use .get() with empty string default to handle missing fields gracefully
        # This ensures consistent sorting behavior even with incomplete data
        return sorted(data, key=lambda x: x.get(params['field'], ''))
```

### Different documentation styles


In [23]:
simple_function = '''
def calculate_discount(price, discount_percent, is_member=False):
    base_discount = price * (discount_percent / 100)
    if is_member:
        base_discount *= 1.1
    final_price = price - base_discount
    return max(0, final_price)
'''

In [24]:
doc_styles = [
    ("Google Style", "Use Google-style docstrings"),
    ("NumPy Style", "Use NumPy-style docstrings"),
    ("Sphinx Style", "Use Sphinx/reStructuredText style docstrings")
]

print("Different Documentation Styles:\n")

for style_name, instruction in doc_styles:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": f"""{instruction} for this function:

```python
{simple_function}
```

Just show the function with the docstring, no explanation."""
        }]
    )
    
    print(f" {style_name}:")
    show_response(response)
    print("\n" + "="*60 + "\n")

Different Documentation Styles:

 Google Style:


```python
def calculate_discount(price, discount_percent, is_member=False):
    """Calculates the final price after applying a discount.
    
    Args:
        price (float): The original price of the item.
        discount_percent (float): The discount percentage to apply (0-100).
        is_member (bool, optional): Whether the customer is a member for additional
            discount. Defaults to False.
    
    Returns:
        float: The final price after discount, minimum value of 0.
    """
    base_discount = price * (discount_percent / 100)
    if is_member:
        base_discount *= 1.1
    final_price = price - base_discount
    return max(0, final_price)
```



 NumPy Style:


```python
def calculate_discount(price, discount_percent, is_member=False):
    """
    Calculate the final price after applying a discount with optional member bonus.

    Parameters
    ----------
    price : float
        The original price of the item.
    discount_percent : float
        The discount percentage to apply (0-100).
    is_member : bool, optional
        Whether the customer is a member (gets 10% bonus discount), by default False

    Returns
    -------
    float
        The final price after discount, minimum value of 0.

    Examples
    --------
    >>> calculate_discount(100, 20)
    80.0
    >>> calculate_discount(100, 20, is_member=True)
    78.0
    """
    base_discount = price * (discount_percent / 100)
    if is_member:
        base_discount *= 1.1
    final_price = price - base_discount
    return max(0, final_price)
```



 Sphinx Style:


```python
def calculate_discount(price, discount_percent, is_member=False):
    """
    Calculate the final price after applying a discount.

    :param price: The original price of the item
    :type price: float
    :param discount_percent: The discount percentage to apply (0-100)
    :type discount_percent: float
    :param is_member: Whether the customer is a member (gets 10% bonus discount)
    :type is_member: bool
    :returns: The final price after discount, minimum 0
    :rtype: float

    .. note::
       Members receive an additional 10% bonus on their discount.

    .. example::
       >>> calculate_discount(100.0, 20.0, True)
       78.0
    """
    base_discount = price * (discount_percent / 100)
    if is_member:
        base_discount *= 1.1
    final_price = price - base_discount
    return max(0, final_price)
```

Generate a README for a project

In [25]:
project_code = '''
# main.py
from dataclasses import dataclass
import requests

@dataclass
class WeatherData:
    temperature: float
    humidity: int
    description: str

class WeatherClient:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://api.weather.com/v1"
    
    def get_current(self, city: str) -> WeatherData:
        response = requests.get(f"{self.base_url}/current", params={"city": city, "key": self.api_key})
        data = response.json()
        return WeatherData(data["temp"], data["humidity"], data["desc"])
    
    def get_forecast(self, city: str, days: int = 5) -> list[WeatherData]:
        response = requests.get(f"{self.base_url}/forecast", params={"city": city, "days": days, "key": self.api_key})
        return [WeatherData(d["temp"], d["humidity"], d["desc"]) for d in response.json()["days"]]
'''

In [26]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    messages=[{
        "role": "user",
        "content": f"""Generate a professional README.md for this Python library:

```python
{project_code}
```

Include:
1. Project title and description
2. Installation instructions
3. Quick start example
4. API reference
5. Contributing section
6. License placeholder"""
    }]
)

show_response(response)

# WeatherPy

A simple and intuitive Python library for fetching weather data from weather APIs. WeatherPy provides a clean interface to retrieve current weather conditions and forecasts for any city.

## Features

- 🌡️ Get current weather conditions
- 📅 Retrieve weather forecasts (up to multiple days)
- 🎯 Simple, type-safe API with dataclasses
- 🔧 Easy to integrate and use

## Installation

```bash
pip install weatherpy
```

### Requirements

- Python 3.9+
- requests

## Quick Start

```python
from weatherpy import WeatherClient

# Initialize the client with your API key
client = WeatherClient(api_key="your-api-key-here")

# Get current weather
current_weather = client.get_current("New York")
print(f"Temperature: {current_weather.temperature}°F")
print(f"Humidity: {current_weather.humidity}%")
print(f"Description: {current_weather.description}")

# Get 7-day forecast
forecast = client.get_forecast("London", days=7)
for day in forecast:
    print(f"Temp: {day.temperature}°F, {day.description}")
```

## API Reference

### WeatherData

A dataclass representing weather information.

```python
@dataclass
class WeatherData:
    temperature: float  # Temperature in Fahrenheit
    humidity: int      # Humidity percentage (0-100)
    description: str   # Weather description (e.g., "Sunny", "Cloudy")
```

### WeatherClient

The main client class for interacting with the weather API.

#### `__init__(api_key: str)`

Initialize the WeatherClient with your API key.

**Parameters:**
- `api_key` (str): Your weather API key

**Example:**
```python
client = WeatherClient(api_key="your-api-key-here")
```

#### `get_current(city: str) -> WeatherData`

Get current weather conditions for a specified city.

**Parameters:**
- `city` (str): Name of the city

**Returns:**
- `WeatherData`: Current weather information

**Example:**
```python
weather = client.get_current("Tokyo")
print(f"Current temperature in Tokyo: {weather.temperature}°F")
```

#### `get_forecast(city: str, days: int = 5) -> list[WeatherData]`

Get weather forecast for a specified city.

**Parameters:**
- `city` (str): Name of the city
- `days` (int, optional): Number of forecast days (default: 5)

**Returns:**
- `list[WeatherData]`: List of weather forecasts

**Example:**
```python
forecast = client.get_forecast("Paris", days=3)
for i, day in enumerate(forecast, 1):
    print(f"Day {i}: {day.temperature}°F - {day.description}")
```

## Error Handling

The library uses the `requests` library under the hood. Make sure to handle potential exceptions:

```python
import requests

try:
    weather = client.get_current("InvalidCity")
except requests.exceptions.RequestException as e:
    print(f"API request failed: {e}")
except KeyError as e:
    print(f"Unexpected API response format: {e}")
```

## Configuration

### API Key Setup

You'll need a valid API key from a weather service provider. The library is configured to work with APIs that follow the standard weather data format.

### Environment Variables

For security, consider storing your API key in environment variables:

```python
import os
from weatherpy import WeatherClient

api_key = os.getenv("WEATHER_API_KEY")
client = WeatherClient(api_key=api_key)
```

## Contributing

We welcome contributions! Here's how you can help:

1. **Fork the repository**
2. **Create a feature branch**: `git checkout -b feature/amazing-feature`
3. **Make your changes** and add tests
4. **Run the tests**: `python -m pytest`
5. **Commit your changes**: `git commit -m 'Add amazing feature'`
6. **Push to the branch**: `git push origin feature/amazing-feature`
7. **Open a Pull Request**

### Development Setup

```bash
# Clone the repository
git clone https://github.com/yourusername/weatherpy.git
cd weatherpy

# Install development dependencies
pip install -e ".[dev]"

# Run tests
python -m pytest

# Run linting
flake8 weatherpy/
black weatherpy/
```

### Reporting Issues

If you find a bug or have a feature request, please open an issue on our [GitHub repository](https://github.com/yourusername/weatherpy/issues).

## License

This project is licensed under the MIT License - see the [LICENSE](LICENSE) file for details.

## Changelog

### v1.0.0
- Initial release
- Current weather data retrieval
- Weather forecast functionality
- Type-safe API with dataclasses

---

**Note**: This library requires a valid API key from a compatible weather service provider. Please refer to your weather API provider's documentation for authentication details.

## Advanced Code Tasks

Converting Python to Javascript

In [27]:
python_code = '''
class ShoppingCart:
    def __init__(self):
        self.items = []
    
    def add_item(self, name, price, quantity=1):
        self.items.append({
            "name": name,
            "price": price,
            "quantity": quantity
        })
    
    def get_total(self):
        return sum(item["price"] * item["quantity"] for item in self.items)
    
    def apply_discount(self, percent):
        discount = self.get_total() * (percent / 100)
        return self.get_total() - discount
'''

In [28]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{
        "role": "user",
        "content": f"""Convert this Python class to modern JavaScript (ES6+):

```python
{python_code}
```

Use classes, arrow functions, and modern JS best practices."""
    }]
)

show_response(response)

Here's the Python class converted to modern JavaScript (ES6+):

```javascript
class ShoppingCart {
  constructor() {
    this.items = [];
  }

  addItem(name, price, quantity = 1) {
    this.items.push({
      name,
      price,
      quantity
    });
  }

  getTotal() {
    return this.items.reduce((total, item) => total + (item.price * item.quantity), 0);
  }

  applyDiscount(percent) {
    const total = this.getTotal();
    const discount = total * (percent / 100);
    return total - discount;
  }
}

// Example usage:
const cart = new ShoppingCart();
cart.addItem("Apple", 1.50, 3);
cart.addItem("Banana", 0.75, 2);

console.log(`Total: $${cart.getTotal()}`); // Total: $6.00
console.log(`With 10% discount: $${cart.applyDiscount(10)}`); // With 10% discount: $5.40
```

**Key modern JavaScript features used:**

1. **ES6 Classes** - Direct equivalent to Python classes
2. **Arrow functions** - Used in the `reduce` method for concise syntax
3. **Default parameters** - `quantity = 1` works the same as Python
4. **Object property shorthand** - `{ name, price, quantity }` instead of `{ name: name, price: price, quantity: quantity }`
5. **Method shorthand** - No need for `function` keyword in class methods
6. **Array.reduce()** - Modern alternative to Python's `sum()` with generator expression
7. **Const/let** - Using `const` for immutable references

**Additional modern enhancements you could consider:**

```javascript
class ShoppingCart {
  #items = []; // Private field

  addItem(name, price, quantity = 1) {
    // Input validation
    if (typeof price !== 'number' || price < 0) {
      throw new Error('Price must be a positive number');
    }
    
    this.#items.push({ name, price, quantity });
  }

  getTotal = () => {
    return this.#items.reduce((total, item) => total + (item.price * item.quantity), 0);
  }

  applyDiscount = (percent) => {
    const total = this.getTotal();
    const discount = total * (percent / 100);
    return Math.round((total - discount) * 100) / 100; // Round to 2 decimal places
  }

  get itemCount() {
    return this.#items.length;
  }
}
```

This enhanced version includes private fields (`#items`), arrow function methods, getters, input validation, and proper number rounding for currency calculations.

### Generating Unit Tests


In [29]:
function_to_test = '''
def validate_password(password: str) -> dict:
    """
    Validate a password against security rules.
    Returns a dict with 'valid' (bool) and 'errors' (list).
    """
    errors = []
    
    if len(password) < 8:
        errors.append("Password must be at least 8 characters")
    
    if not any(c.isupper() for c in password):
        errors.append("Password must contain at least one uppercase letter")
    
    if not any(c.islower() for c in password):
        errors.append("Password must contain at least one lowercase letter")
    
    if not any(c.isdigit() for c in password):
        errors.append("Password must contain at least one number")
    
    if not any(c in "!@#$%^&*" for c in password):
        errors.append("Password must contain at least one special character (!@#$%^&*)")
    
    return {"valid": len(errors) == 0, "errors": errors}
'''

In [30]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    messages=[{
        "role": "user",
        "content": f"""Generate comprehensive pytest unit tests for this function:

```python
{function_to_test}
```

Include:
1. Test valid passwords
2. Test each error condition
3. Edge cases
4. Parametrized tests where appropriate"""
    }]
)

show_response(response)

```python
import pytest


def validate_password(password: str) -> dict:
    """
    Validate a password against security rules.
    Returns a dict with 'valid' (bool) and 'errors' (list).
    """
    errors = []

    if len(password) < 8:
        errors.append("Password must be at least 8 characters")

    if not any(c.isupper() for c in password):
        errors.append("Password must contain at least one uppercase letter")

    if not any(c.islower() for c in password):
        errors.append("Password must contain at least one lowercase letter")

    if not any(c.isdigit() for c in password):
        errors.append("Password must contain at least one number")

    if not any(c in "!@#$%^&*" for c in password):
        errors.append("Password must contain at least one special character (!@#$%^&*)")

    return {"valid": len(errors) == 0, "errors": errors}


class TestValidatePassword:
    """Test cases for the validate_password function."""

    def test_valid_passwords(self):
        """Test passwords that meet all requirements."""
        valid_passwords = [
            "Password1!",
            "MySecure123@",
            "StrongP@ss1",
            "Complex9#Pass",
            "Aa1!bcdefgh",  # Minimum length with all requirements
            "VeryLongPassword123!",  # Longer password
            "Mixed123!@#Cases",  # Multiple special chars
            "aA1!bcdefghijklmnop",  # Much longer password
        ]
        
        for password in valid_passwords:
            result = validate_password(password)
            assert result["valid"] is True, f"Password '{password}' should be valid"
            assert result["errors"] == [], f"Password '{password}' should have no errors"

    def test_empty_password(self):
        """Test empty password fails all validations."""
        result = validate_password("")
        expected_errors = [
            "Password must be at least 8 characters",
            "Password must contain at least one uppercase letter",
            "Password must contain at least one lowercase letter",
            "Password must contain at least one number",
            "Password must contain at least one special character (!@#$%^&*)"
        ]
        assert result["valid"] is False
        assert result["errors"] == expected_errors

    @pytest.mark.parametrize("password,expected_error", [
        ("Pass1!", "Password must be at least 8 characters"),
        ("a", "Password must be at least 8 characters"),
        ("1234567", "Password must be at least 8 characters"),
        ("Aa1!abc", "Password must be at least 8 characters"),  # 7 chars, otherwise valid
    ])
    def test_length_validation(self, password, expected_error):
        """Test passwords that are too short."""
        result = validate_password(password)
        assert result["valid"] is False
        assert expected_error in result["errors"]

    @pytest.mark.parametrize("password", [
        "password1!",  # No uppercase
        "alllowercase123!",  # No uppercase
        "nouppercasehere9@",  # No uppercase
        "123456789!",  # No letters at all
    ])
    def test_missing_uppercase(self, password):
        """Test passwords missing uppercase letters."""
        result = validate_password(password)
        assert result["valid"] is False
        assert "Password must contain at least one uppercase letter" in result["errors"]

    @pytest.mark.parametrize("password", [
        "PASSWORD1!",  # No lowercase
        "ALLUPPERCASE123!",  # No lowercase
        "NOLOWERCASEHERE9@",  # No lowercase
        "123456789!",  # No letters at all
    ])
    def test_missing_lowercase(self, password):
        """Test passwords missing lowercase letters."""
        result = validate_password(password)
        assert result["valid"] is False
        assert "Password must contain at least one lowercase letter" in result["errors"]

    @pytest.mark.parametrize("password", [
        "Password!",  # No digits
        "NoNumbers!@#",  # No digits
        "OnlyLetters!",  # No digits
        "AbCdEfGh!@#$",  # No digits
    ])
    def test_missing_digits(self, password):
        """Test passwords missing digits."""
        result = validate_password(password)
        assert result["valid"] is False
        assert "Password must contain at least one number" in result["errors"]

    @pytest.mark.parametrize("password", [
        "Password1",  # No special chars
        "NoSpecialChars123",  # No special chars
        "OnlyAlphaNum987",  # No special chars
        "Password123-",  # Dash is not in allowed special chars
        "Password123_",  # Underscore is not in allowed special chars
        "Password123+",  # Plus is not in allowed special chars
    ])
    def test_missing_special_characters(self, password):
        """Test passwords missing required special characters."""
        result = validate_password(password)
        assert result["valid"] is False
        assert "Password must contain at least one special character (!@#$%^&*)" in result["errors"]

    @pytest.mark.parametrize("special_char", ["!", "@", "#", "$", "%", "^", "&", "*"])
    def test_each_special_character_accepted(self, special_char):
        """Test that each allowed special character is accepted."""
        password = f"Password1{special_char}"
        result = validate_password(password)
        assert result["valid"] is True
        assert result["errors"] == []

    def test_multiple_validation_failures(self):
        """Test password that fails multiple validations."""
        password = "short"  # Too short, no uppercase, no digits, no special chars
        result = validate_password(password)
        expected_errors = [
            "Password must be at least 8 characters",

### Debugging Assistance


Debug code with an error

In [31]:
buggy_code = '''
def find_duplicates(items):
    """Find duplicate items in a list."""
    seen = {}
    duplicates = []
    
    for item in items:
        if item in seen:
            duplicates.append(item)
        seen[item] = True
    
    return duplicates

# Expected: [2, 3]
# Actual: [2, 2, 3]
result = find_duplicates([1, 2, 2, 2, 3, 3, 4])
'''

In [32]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{
        "role": "user",
        "content": f"""Debug this code. The expected output is [2, 3] but it returns [2, 2, 3]:

```python
{buggy_code}
```

1. Identify the bug
2. Explain why it happens
3. Provide the fixed code"""
    }]
)

show_response(response)

## 1. Identify the bug

The bug is that the function adds an item to the `duplicates` list **every time** it encounters a duplicate, not just the **first time** it identifies an item as a duplicate.

## 2. Explain why it happens

Let's trace through the execution with input `[1, 2, 2, 2, 3, 3, 4]`:

- `1`: Not in `seen` → add to `seen`
- `2`: Not in `seen` → add to `seen`
- `2`: **In `seen`** → add to `duplicates` → `duplicates = [2]`
- `2`: **In `seen`** → add to `duplicates` → `duplicates = [2, 2]` ❌
- `3`: Not in `seen` → add to `seen`
- `3`: **In `seen`** → add to `duplicates` → `duplicates = [2, 2, 3]`
- `4`: Not in `seen` → add to `seen`

The issue is that when we encounter the third `2`, it's already in `seen`, so it gets added to `duplicates` again. We only want to add each duplicate value once.

## 3. Fixed code

Here are two solutions:

### Solution 1: Track if we've already added the duplicate
```python
def find_duplicates(items):
    """Find duplicate items in a list."""
    seen = {}
    duplicates = []

    for item in items:
        if item in seen:
            # Only add to duplicates if this is the first time we've seen a duplicate
            if seen[item] == 1:  # First duplicate occurrence
                duplicates.append(item)
            seen[item] += 1
        else:
            seen[item] = 1

    return duplicates
```

### Solution 2: Use a set to track added duplicates
```python
def find_duplicates(items):
    """Find duplicate items in a list."""
    seen = set()
    duplicates = set()

    for item in items:
        if item in seen:
            duplicates.add(item)
        else:
            seen.add(item)

    return list(duplicates)
```

Both solutions will correctly return `[2, 3]` for the input `[1, 2, 2, 2, 3, 3, 4]`.

**Note:** Solution 2 using sets won't preserve the order of first duplicate occurrence. If order matters, use Solution 1.

<center>
     <img src="https://raw.githubusercontent.com/DataForScience/Networks/master/data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>